## Projeto Sprint 8 - Análise de Negócio

### Contexto do Projeto
Trabalhamos no departamento analítico da Y.Afisha e fomos encarregados de otimizar os investimentos em marketing da empresa. Para isso, precisamos entender melhor o perfil dos consumidores e a eficiência dos diferentes canais de aquisição.

### Objetivo
Analisar os dados de janeiro de 2017 a dezembro de 2018 para otimizar os gastos com marketing e maximizar o retorno sobre investimento (ROI).

### Perguntas de Negócio
Este projeto busca responder às seguintes questões estratégicas:
* Qual é o perfil do nosso consumidor?
* Por quais canais de marketing nossos clientes chegam até nós?
* Quando os usuários começam a fazer compras após o primeiro acesso?
* Qual é o retorno sobre investimento (ROI) dos nossos canais de marketing?
* Quanto tempo demora para recuperar o investimento em aquisição de clientes?
* Qual canal oferece o melhor custo-benefício para futuros investimentos?

In [1]:
# Bibliotecas
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns   
import numpy as np



In [2]:
# Configurações de visualização
plt.style.use('default')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 10

In [3]:
#Importando os dados de visitas(visits) já com a conversão nas colunas referentes a Start Ts e End Ts (object -> datetime).
# Convertendo a coluna Device para 'category' visto que só temos 2 categorias (Touch e Desktop) para que futuramente possa ser utilizada de forma mais eficiente.
#Além disso, convertendo os nomes das colunas para letras minúsculas para facilitar a manipulação dos dados.
visits = pd.read_csv('visits_log_us.csv', 
                     parse_dates=['Start Ts', 'End Ts'], 
                     dtype={'Device': 'category'})
visits.columns = visits.columns.str.lower()

In [4]:
#importando os dados de pedidos (order) já com a conversão na coluna Buy Ts (object -> datetime).
#Convertendo os nomes das colunas para letras minúsculas para facilitar a manipulação dos dados.
#Convertendo os nomes das colunas para letras minúsculas para facilitar a manipulação dos dados.
order = pd.read_csv('orders_log_us.csv',
                     parse_dates=['Buy Ts'])
order.columns = order.columns.str.lower()

In [5]:
#Importando os dados de custos(costs) já com a conversão na coluna Dt de (object -> datetime).
#Convertendo os nomes das colunas para letras minúsculas para facilitar a manipulação dos dados.
#Convertendo os nomes das colunas para letras minúsculas para facilitar a manipulação dos dados.
costs = pd.read_csv('costs_us.csv',
                     parse_dates=['dt'])
costs.columns = costs.columns.str.lower()

## Análisando os DataFrames 

In [6]:
#Analisando os dados de visitas.
visits.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 359400 entries, 0 to 359399
Data columns (total 5 columns):
 #   Column     Non-Null Count   Dtype         
---  ------     --------------   -----         
 0   device     359400 non-null  category      
 1   end ts     359400 non-null  datetime64[ns]
 2   source id  359400 non-null  int64         
 3   start ts   359400 non-null  datetime64[ns]
 4   uid        359400 non-null  uint64        
dtypes: category(1), datetime64[ns](2), int64(1), uint64(1)
memory usage: 11.3 MB


In [7]:
#Observando os dados de visitas(visits).
visits.head()

,device,end ts,source id,start ts,uid
0,touch,2017-12-20 17:38:00,4,2017-12-20 17:20:00,16879256277535980062
1,desktop,2018-02-19 17:21:00,2,2018-02-19 16:53:00,104060357244891740
2,touch,2017-07-01 01:54:00,5,2017-07-01 01:54:00,7459035603376831527
3,desktop,2018-05-20 11:23:00,9,2018-05-20 10:59:00,16174680259334210214
4,desktop,2017-12-27 14:06:00,3,2017-12-27 14:06:00,9969694820036681168


In [8]:
#Analisando os dados de pedidos(order).
order.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50415 entries, 0 to 50414
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype         
---  ------   --------------  -----         
 0   buy ts   50415 non-null  datetime64[ns]
 1   revenue  50415 non-null  float64       
 2   uid      50415 non-null  uint64        
dtypes: datetime64[ns](1), float64(1), uint64(1)
memory usage: 1.2 MB


In [9]:
#Observando os dados de vendas(order).
order.head()

,buy ts,revenue,uid
0,2017-06-01 00:10:00,17.00,10329302124590727494
1,2017-06-01 00:25:00,0.55,11627257723692907447
2,2017-06-01 00:27:00,0.37,17903680561304213844
3,2017-06-01 00:29:00,0.55,16109239769442553005
4,2017-06-01 07:58:00,0.37,14200605875248379450


In [10]:
#Analisando os dados de custos(costs).
costs.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2542 entries, 0 to 2541
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   source_id  2542 non-null   int64         
 1   dt         2542 non-null   datetime64[ns]
 2   costs      2542 non-null   float64       
dtypes: datetime64[ns](1), float64(1), int64(1)
memory usage: 59.7 KB


In [11]:
costs.head()

,source_id,dt,costs
0,1,2017-06-01,75.20
1,1,2017-06-02,62.25
2,1,2017-06-03,36.53
3,1,2017-06-04,55.00
4,1,2017-06-05,57.08


### Análisando Coortes Comportamentais - Visits

In [12]:
#Criando as colunas daily_visits que representa o dia que o usuário acessou.
visits['daily_visits'] = visits['start ts'].dt.date
#Criando as colunas week_visits que representa a semana que o usuário acessou.
visits['week_visits'] = visits['start ts'].dt.isocalendar().week
#Criando as colunas month_visits que representa o mês que o usuário acessou.
visits['month_visits'] = visits['start ts'].dt.month    
#Criando as colunas year_visits que representa o ano que o usuário acessou.
visits['year_visits'] = visits['start ts'].dt.year  

In [13]:
#Calculando o número de visitas diárias por usuário, agrupando por daily_visits e contando o número de usuários únicos (uid) para cada dia, e depois contando o número total de visitas para cada dia.
visits_per_day = visits.groupby(['daily_visits']).agg({'uid':['nunique', 'count']}).reset_index()
#renomeando as colunas para facilitar a manipulação dos dados.
visits_per_day.columns = ['daily_visits', 'unique_users', 'total_visits']
#Observando os dados de visitas por usuário.
visits_per_day.head()

,daily_visits,unique_users,total_visits
0,2017-06-01,605,664
1,2017-06-02,608,658
2,2017-06-03,445,477
3,2017-06-04,476,510
4,2017-06-05,820,893


In [14]:
#Calculando quantidade de visitas por semana, agrupando por week_visits e contando o número de usuários únicos (uid) para cada semana, incluindo o ano, e depois contando o número total de visitas para cada semana.
visits_per_week = visits.groupby(['week_visits','year_visits']).agg({'uid':'nunique'}).reset_index()
#renomeando as colunas para facilitar a manipulação dos dados. 
visits_per_week.columns = ['week_visits','year_visits', 'unique_users']
#Observando os dados de visitas por semana.
visits_per_week.head()

,week_visits,year_visits,unique_users
0,1,2018,6918
1,2,2018,6703
2,3,2018,6972
3,4,2018,7060
4,5,2018,8111


In [15]:
#calculando o numero de visitas por mês, agrupando por month_visits e contando o número de usuários únicos (uid) para cada mês, incluindo o ano, e depois contando o número total de visitas para cada mês.
visits_per_month = visits.groupby(['month_visits','year_visits']).agg({'uid':'nunique'}).reset_index()
#renomeando as colunas para facilitar a manipulação dos dados.              
visits_per_month.columns = ['month_visits','year_visits', 'unique_users']
#Observando os dados de visitas por mês.
visits_per_month.head()

,month_visits,year_visits,unique_users
0,1,2018,28716
1,2,2018,28749
2,3,2018,27473
3,4,2018,21008
4,5,2018,20701


In [16]:
#Comprimento de cada sessao de visitas, subtraindo a coluna 'start ts' da coluna 'end ts' para obter a duração de cada sessão em segundos, e depois convertendo para horas dividindo por 3600.
session_length = (visits['end ts'] - visits['start ts']).dt.total_seconds() / 3600
#Apenas usuario que tiveram sessions_length maior que 1 minuto (0.0167 horas) e menor que 3 horas, para considerar apenas sessões ativas.
active_sessions = visits[(session_length > 0.0167) & (session_length < 3)]
#Adicionando a coluna session_length ao dataframe de visitas(visits) e arredondando os valores para 2 casas decimais para melhor visualização.
visits['session_length'] = session_length.round(2)
visits.head()


,device,end ts,source id,start ts,uid,daily_visits,week_visits,month_visits,year_visits,session_length
0,touch,2017-12-20 17:38:00,4,2017-12-20 17:20:00,16879256277535980062,2017-12-20,51,12,2017,0.30
1,desktop,2018-02-19 17:21:00,2,2018-02-19 16:53:00,104060357244891740,2018-02-19,8,2,2018,0.47
2,touch,2017-07-01 01:54:00,5,2017-07-01 01:54:00,7459035603376831527,2017-07-01,26,7,2017,0.00
3,desktop,2018-05-20 11:23:00,9,2018-05-20 10:59:00,16174680259334210214,2018-05-20,20,5,2018,0.40
4,desktop,2017-12-27 14:06:00,3,2017-12-27 14:06:00,9969694820036681168,2017-12-27,52,12,2017,0.00


In [17]:
#Sessoes por usuario, agrupando por uid e contando o número de sessões para cada usuário.
sessions_per_user = visits.groupby('uid').agg({'daily_visits':'count'}).reset_index()
#Frequencia que os usuarios voltam ao app
returning_users = (sessions_per_user[sessions_per_user['daily_visits'] > 1])
#Calculando a taxa de retorno dos usuários, dividindo o número de usuários que retornaram pelo número total de usuários e multiplicando por 100 para obter a porcentagem.
users_total = len(sessions_per_user)
users_returning = len(returning_users)
taxa_retorno = (users_returning / users_total) * 100
print(f'Taxa de retorno dos usuários: {taxa_retorno:.2f}%')

Taxa de retorno dos usuários: 22.85%
